# 01. Data Collection (USGS API -> MongoDB)

Notebook ini bertugas untuk menyedot data mentah gempa bumi dari **USGS Earthquake API** dan langsung menyimpannya ke **MongoDB**.
Pengambilan data akan dipecah per 15 hari untuk menghindari batas maksimal (*limit*) API sebanyak 20.000 data per panggilan.

### Tahap 1: Persiapan Environment
Memuat library yang dibutuhkan dan mengambil pengaturan dari file `.env` di luar folder.

In [ ]:
import os
import requests
import time
from datetime import datetime, timedelta
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv('../.env')
MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017')
MONGO_DB = os.getenv('MONGO_DB', 'earthquake_db')
MONGO_RAW_COL = os.getenv('MONGO_RAW_COLLECTION', 'raw_earthquakes')
USGS_URL = os.getenv('USGS_BASE_URL', 'https://earthquake.usgs.gov/fdsnws/event/1/query')
DATA_YEAR = int(os.getenv('DATA_YEAR', '2025'))

### Tahap 2: Membuka Koneksi MongoDB
Menghubungkan *Python* ke *Database MongoDB* dan menentukan koleksi target penampung data mentah.

In [ ]:
client = MongoClient(MONGO_URI)
db = client[MONGO_DB]
collection = db[MONGO_RAW_COL]

print(f"Terhubung ke MongoDB: {MONGO_URI}")
print(f"Target Koleksi: {MONGO_DB}.{MONGO_RAW_COL}")

### Tahap 3: Fungsi Penarik Data per Periode (Chunk)
Fungsi ini akan mengambil data dari API, mengecek apakah terjadi *error* (limit tercapai), dan langsung menyimpannya ke MongoDB (Append).

In [ ]:
def fetch_data_chunk(start_date, end_date):
    params = {
        'format': 'geojson',
        'starttime': start_date,
        'endtime': end_date
    }
    try:
        print(f"Menarik data {start_date} s/d {end_date}...", end=' ')
        res = requests.get(USGS_URL, params=params, timeout=30)
        
        if res.status_code == 400:
            print("Gagal: Hit limit/Bad Request dari USGS.")
            return 0
            
        res.raise_for_status()
        data = res.json()
        features = data.get('features', [])
        
        if features:
            collection.insert_many(features)
            print(f"Tersimpan {len(features)} data.")
            return len(features)
        else:
            print("Tidak ada gempa.")
            return 0
            
    except Exception as e:
        print(f"Error: {e}")
        return 0

### Tahap 4: Eksekusi Penarikan Data (Setahun Penuh)
Memecah 365 hari dalam setahun menjadi *chunks* per 15 hari. Fungsi akan berhenti sejenak tiap putarannya (Time Sleep) untuk menghormati batasan sistem API.

In [ ]:
total_saved = 0
start_dt = datetime(DATA_YEAR, 1, 1)
end_dt = datetime(DATA_YEAR, 12, 31)
current_dt = start_dt

while current_dt <= end_dt:
    next_dt = current_dt + timedelta(days=15)
    if next_dt > end_dt:
        next_dt = end_dt
        
    str_start = current_dt.strftime('%Y-%m-%d')
    str_end = next_dt.strftime('%Y-%m-%d')
    
    total_saved += fetch_data_chunk(str_start, str_end)
    current_dt = next_dt + timedelta(days=1)
    time.sleep(1)

print(f"\nPROSES SELESAI! Total data sukses tersimpan: {total_saved}")